# Creation of the Dataset for the prediction task
- We want to create a dataset where we create some features that could be useful for the task of classifying wether a cyclist is going to place in the top20 or not (binary classification).
- As we noticed in the previous tasks, the history of a cyclist is an important factor to have in order to obtain knowledge and we think that it is going also to be crucial for this kind of classification.
- We want to create features that represent the history of the cyclists of his "previous races". For previous races we mean the races made by the cyclist before the race we are taking into consideration, for example:
    - Consider a cyclist that made 3 races, one for each year (2000,2001,2002), when we create the features for the cyclist in the year 2002 we will take into consideration the races made in 2000 and 2001, when we consider the race made in 2000 we do not have this kind of information and the feature will result NaN.

# Features created
- <span style="color: red;">avg_position</span>: The average position of a cyclist in the previous races
- <span style="color: red;">avg_position_x</span>: The average position of a cyclist in the previous x races, when x races are not available, just consider the maximum possible amount, but still need a minumum given by x-1.
- <span style="color: red;">races_amount</span>: The amount of races made by a cyclist
- <span style="color: red;">performance_wma</span>: moving average of the position of previous races, it considers a window of 5 previous races
- <span style="color: red;">top20 consistency</span>: it is the amount of times a cyclist ended up in top20 in the previous races
- <span style="color: red;">races_dby_consistency</span>: top20 consistency divided by races_amount, to avoid division by 0, top20_series is set to 1 when is 0
- <span style="color: red;">prev_position_x</span>: the position of the cyclist x races ago.


# Some adjustements in the normalized version:
-  <span style="color: red;"> Position</span>: now position is normalized with respect to the number of participants.
- <span style="color: red;"> position_by_startlist</span>: a new position which takes into account the startlist to normalize the position (higher startlist, lower position score).
- <span style="color: red;"> avg_position_startlist</span>: the average of the previous races but using position_by_startlist.
- <span style="color: red;"> avg_position_x_startlist</span>: the average of the previous x races but using position_by_startlist.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import gzip
import numpy as np

save = True
save2 = True

In [2]:
# Take the merged dataset cleaned from null values and anomalies
with gzip.open('../filtered_clean_merged_df.pkl', 'r') as file:
    filtered_clean_dataset = pickle.load(file)

In [3]:
# create top20 label
filtered_clean_dataset['top20'] = (filtered_clean_dataset['position'] <= 20).astype(int)

In [4]:
# Sort the dataset by date
df_ordered_by_date = filtered_clean_dataset.sort_values(['date'])

In [5]:
# Create a new column with the number of participants

df_ordered_by_date_races = df_ordered_by_date.groupby(['race_id']).size().reset_index(name='participants')
df_ordered_by_date = pd.merge(df_ordered_by_date, df_ordered_by_date_races)

In [6]:
# normalize the position (position / startlist quality)

df_ordered_by_date['position_by_startlist'] = df_ordered_by_date['position'] / df_ordered_by_date['startlist_quality']

In [7]:
# normalize the position (position / number of cyclists)

df_ordered_by_date['position'] = df_ordered_by_date['position'] / df_ordered_by_date['participants']

In [8]:
races = df_ordered_by_date.groupby('cyclist')['position']
races_startlist = df_ordered_by_date.groupby('cyclist')['position_by_startlist']
# Create avg_position

for group_name,group_data in races:
    avg_positions = group_data.expanding().mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position'] = avg_positions


In [9]:
# Create avg_position_startlist

for group_name,group_data in races_startlist:
    avg_positions_startlist = group_data.expanding().mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_startlist'] = avg_positions_startlist

Normalized by participants average positions

In [10]:
# Create avg_position_1
for group_name,group_data in races:
    avg_positions_1 = group_data.rolling(window=1, min_periods=1).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_1'] = avg_positions_1

In [11]:
# Create avg_position_2
for group_name,group_data in races:
    avg_positions_2 = group_data.rolling(window=2, min_periods=1).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_2'] = avg_positions_2

In [12]:
# Create avg_position_3
for group_name,group_data in races:
    avg_positions_3 = group_data.rolling(window=3, min_periods=2).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_3'] = avg_positions_3

In [13]:
# Create avg_position_4
for group_name,group_data in races:
    avg_positions_4 = group_data.rolling(window=4, min_periods=3).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_4'] = avg_positions_4

Startlist based average positions

In [14]:
# Create avg_position_1_startlist
for group_name,group_data in races_startlist:
    avg_positions_1_startlist = group_data.rolling(window=1, min_periods=1).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_1_startlist'] = avg_positions_1_startlist

In [15]:
# Create avg_position_2_startlist
for group_name,group_data in races_startlist:
    avg_positions_2_startlist = group_data.rolling(window=2, min_periods=1).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_2_startlist'] = avg_positions_2_startlist

In [16]:
# Create avg_position_3_startlist
for group_name,group_data in races_startlist:
    avg_positions_3_startlist = group_data.rolling(window=3, min_periods=2).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_3_startlist'] = avg_positions_3_startlist

In [17]:
# Create avg_position_4_startlist
for group_name,group_data in races_startlist:
    avg_positions_4_startlist = group_data.rolling(window=4, min_periods=3).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_4_startlist'] = avg_positions_4_startlist

In [18]:
# Create races_amount
for group_name,group_data in races:
    races_amount = group_data.expanding().count().shift()
    df_ordered_by_date.loc[group_data.index, 'races_amount'] = races_amount

In [19]:
# Weighted moving average
for group_name, group_data in races:
    performance_wma = group_data.ewm(span=5, adjust=False).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'performance_wma'] = performance_wma

In [20]:
# top20 consistency
for group_name, group_data in races:
    top20_series = (group_data <= 20).expanding().sum().shift()
    df_ordered_by_date.loc[group_data.index, 'top20_consistency'] = top20_series

In [21]:
# top20 consistency divided by races_amount, to avoid division by 0, top20_series is set to 1 when is 0
for group_name, group_data in races:
    races_amount = group_data.expanding().count().shift()
    top20_series = (group_data <= 20).expanding().sum().shift()
    top20_series[top20_series == 0] = 1
    races_dby_consistency = races_amount/top20_series
    df_ordered_by_date.loc[group_data.index, 'races_dby_consistency'] = races_dby_consistency

We consider as test set the races made from 2022 onwards.

In [22]:
# Create training set: races until 2021
train_set = df_ordered_by_date[df_ordered_by_date['year'] <= 2021]

# Create test set: races from 2022 onwards
test_set = df_ordered_by_date[df_ordered_by_date['year'] > 2021]

In [23]:
if save:
    with gzip.open('train_set_normalized.pkl', 'wb') as file:
        pickle.dump(train_set, file)
    with gzip.open('test_set_normalized.pkl', 'wb') as file:
        pickle.dump(test_set, file)
if save2:
    with gzip.open('prediction_dataset_normalized.pkl', 'wb') as file:
        pickle.dump(df_ordered_by_date, file)